# Alternative RCM BA - 3. Apply F2 to the four RCM runs

**Alternative solution** (to the supervisor's downscaling pipeline) for the bias adjustment of the RCM:

1. **F1** (`GCM_BA/`, done): WDF + QDM of EC-Earth3 towards ERA5 on the EC-Earth3 grid (0.7 deg), applied to
   `gcm_2025_nohosing` -> the target climate, ERA5-like, at 0.7 deg.
2. **Reference quantiles at 0.25 deg** (BCSD principle, Wood et al. 2004), per calendar month m (the 3 months centred on
   m, as F1) and quantile: Q_ref = Q_ERA5,0.25 + I[Q_F1 - Q_ERA5,0.7] for tasmax, ratios (interpolated in log) for DTR
   and the pr wet days; pr: dry-day fraction first, p0_ref = p0_ERA5,0.25 + I[p0_F1 - p0_ERA5,0.7] (ISIMIP3BASD
   principle, Lange 2019). I = bilinear interpolation from 0.7 to 0.25 deg. The RCM does not enter the reference.
3. **F2**: QDM of `rcm_2025_nohosing` towards Q_ref (`bu.fit_F2`: the xsdba training with Q_ref as ref quantiles),
   applied unchanged to the four RCM runs with `bu.apply_F` (the code of F1), each run ranked in its own distribution.

The RCM keeps its own day-to-day sequence and spatial structure (they carry the AMOC effect): the correction acts on
the distribution of each pixel only.

`bu.apply_F` (the code of F1): xsdba `QuantileDeltaMapping.adjust`, month by month (3-month windows), linear
interpolation between the quantiles, constant extrapolation; pr jittered below 0.1 mm/day then set back to 0. Only
European land kept (`bu.keep_mask`, as F1), sea kept. Products: `RCM_bias_adjusted_0.25deg/<run>_F2_<tag>.nc`.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                       # pr, tasmax, tasmin (tasmin through DTR, as F1)

# ---- F1 (GCM_BA): the same tags as GCM_BA notebooks 2-3
CAL_PERIOD = ("1981", "2014")                 # calibration period of F1 (ERA5 / EC-Earth3 historical)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}_europe"
F1_PRODUCTS = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")

# ---- alternative solution
TARGET_GCM = "gcm_2025_nohosing"              # F1 product that gives the target climate of the RCM calibration
RCM_CAL = "rcm_2025_nohosing"                 # RCM run driven by it: F2 calibrated on it
RCM_RUNS = ["rcm_2025_nohosing", "rcm_2025_hosing", "rcm_2050_nohosing", "rcm_2050_hosing"]
MARGIN = 1.0                                  # degrees around the EFAS domain (LISFLOOD Europe)
REGION = {k: (bu.EFAS_BOX[k][0] - MARGIN, bu.EFAS_BOX[k][1] + MARGIN) for k in ("lat", "lon")}
COARSE_REGION = {k: (REGION[k][0] - 1.0, REGION[k][1] + 1.0) for k in ("lat", "lon")}   # + 1 coarse cell, for the interpolation
CHUNKS_SPACE = {"lat": 32, "lon": 64}
RCM_READY = bu.dataset_available(cfg, RCM_CAL, "pr")


def ref_years():
    """Years of the reference = years of the RCM calibration run (the F1 product over the same years); as long as the
    RCM is not in the config: all the years of the F1 product (to recompute once the RCM arrives)."""
    src = bu.read_raw(cfg, RCM_CAL, "pr") if RCM_READY else bu.read_processed(F1_PRODUCTS / f"{TARGET_GCM}_{F1_TAG}.nc", "pr")
    y = src.time.dt.year.values
    return str(y.min()), str(y.max())


REF_YEARS = ref_years()
QREF_TAG = f"{F1_TAG}_{TARGET_GCM}_{REF_YEARS[0]}-{REF_YEARS[1]}"
QREF_FILE = bu.alt_dir(cfg, "1_reference_quantiles") / f"qref_{QREF_TAG}.nc"
F2_DIR = bu.alt_dir(cfg, "2_F2_calculation", f"F2_{QREF_TAG}")
OUT = bu.alt_dir(cfg, "RCM_bias_adjusted_0.25deg")
print(f"region {REGION} | reference years {REF_YEARS[0]}-{REF_YEARS[1]}"
      + ("" if RCM_READY else " (RCM not in the config yet: all the years of the F1 product)"))
print("reference quantiles:", QREF_FILE, "\nF2:", F2_DIR, "\nproducts:", OUT)

In [ ]:
assert RCM_READY, f"{RCM_CAL}: RCM paths not in the config yet (notebook 00)"
assert (F2_DIR / "qdm_pr.nc").exists(), "F2 not calibrated: run notebook 2 first"
OVERWRITE = False
for name in RCM_RUNS:
    out_path = OUT / f"{name}_F2_{QREF_TAG}.nc"
    if out_path.exists() and not OVERWRITE:
        print(f"[already done] {name}"); continue
    if not all(bu.dataset_available(cfg, name, v) for v in VARS):
        print(f"[missing] {name}: inputs missing (config, notebook 00)"); continue
    sim = {v: bu.crop_bbox(bu.read_raw(cfg, name, v), REGION) for v in VARS}
    t0 = time.time()
    adj = bu.apply_F(sim, F2_DIR, chunks_space=CHUNKS_SPACE)
    adj = {v: a.where(bu.keep_mask(a)) for v, a in adj.items()}   # land outside Europe -> NaN
    bu.save_ds(xr.Dataset(adj), out_path)
    print(f"{name}: {(time.time() - t0) / 3600:.1f} h")

## Quick checks (first 3 years): pr >= 0, tasmin < tasmax, no new NaN

In [ ]:
rows = []
for name in RCM_RUNS:
    f = OUT / f"{name}_F2_{QREF_TAG}.nc"
    if not f.exists():
        continue
    ds = xr.open_dataset(f, chunks={"time": 31}).isel(time=slice(0, 365 * 3))
    raw = bu.crop_bbox(bu.read_raw(cfg, name, "pr"), REGION).isel(time=slice(0, 365 * 3))
    inside = raw.notnull() & bu.keep_mask(raw)
    rows.append({"run": name, "pr min": float(ds["pr"].min()),
                 "frac tasmin >= tasmax (should be 0)": float(((ds["tasmin"] >= ds["tasmax"]) & inside).sum() / inside.sum()),
                 "new NaN (should be 0)": float((ds["pr"].isnull() & inside).sum() / inside.sum())})
display(pd.DataFrame(rows).set_index("run"))